# Trivima — GS-LRM (Universal Image → 3D Gaussian Splat)

Trains a single model that takes any photo (indoor / outdoor street / landmark)
and outputs 3D Gaussians directly in one forward pass.

**Architecture:** DINOv2 ViT-B + DPT decoder + per-pixel Gaussian head.
**Loss:** L1 RGB on rendered novel views + masked L1 depth supervision.

**Phase plan:**
- A. Replica overfit smoke test (30 min, indoor only)
- B. Three-domain smoke test: Replica + KITTI + MegaDepth (3h)
- C. Mid scale: + ScanNet++ + KITTI-360 + MegaScenes (5h)
- D. Production: + nuScenes + Waymo + HM3D + Tanks and Temples (multi-session)

**Drive auto-resume** is built in — if the runtime restarts, just Run all again.

## 1. Install + Drive mount

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q transformers gsplat plyfile h5py

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/trivima_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted: {DRIVE_DIR}')
!ls -lh {DRIVE_DIR}/ 2>/dev/null || echo 'no existing checkpoints'

## 2. Clone trivima repo

In [ ]:
!git clone https://github.com/Harishmelvic/trivima.git /content/trivima 2>/dev/null || (cd /content/trivima && git pull)
import sys
sys.path.insert(0, '/content/trivima')
sys.path.insert(0, '/content/trivima/tools')
print('repo ready')

## 3. Download Replica (zero-friction starter, ~16GB)

In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)

# Replica is distributed via the official download.sh script (not a single zip).
# Total ~16GB across multiple parts. The script is in the Replica-Dataset repo.
if not os.path.exists('/content/data/Replica') or len(os.listdir('/content/data/Replica')) == 0:
    !git clone https://github.com/facebookresearch/Replica-Dataset.git /content/Replica-Dataset 2>/dev/null || true
    !cd /content/Replica-Dataset && bash download.sh /content/data/Replica 2>&1 | tail -20

print('Replica scene folders:')
!ls /content/data/Replica/ 2>/dev/null | head
n_scenes = len([d for d in os.listdir('/content/data/Replica') if os.path.isdir(f'/content/data/Replica/{d}')]) if os.path.exists('/content/data/Replica') else 0
print(f'Total: {n_scenes} scene folders')
if n_scenes == 0:
    print('\\n!!! Replica download failed.')
    print('Workaround: manually upload Replica scenes to /content/data/Replica/')
    print('Or skip ahead to Phase B and use a different starter dataset.')

## 4. Install Habitat-Sim and pack Replica → .torch

In [ ]:
# Habitat-Sim install in Colab is not pip-trivial. The official path uses
# conda. We use condacolab to bootstrap conda inside the Colab kernel.
#
# !!! IMPORTANT: condacolab.install() RESTARTS the runtime once. After it
# restarts, re-run this cell (it will skip the install and proceed to packing).

import os, sys

if not os.path.exists('/usr/local/lib/python3.10/site-packages/habitat_sim') and \
   not os.path.exists('/usr/local/lib/python3.11/site-packages/habitat_sim') and \
   not any('habitat_sim' in p for p in sys.path):
    try:
        import habitat_sim
        print('habitat_sim already installed:', habitat_sim.__version__)
    except ImportError:
        print('Installing habitat-sim via condacolab (will RESTART runtime)...')
        !pip install -q condacolab
        import condacolab
        condacolab.install()  # restarts
        # After restart, the conda env is on PATH; install habitat-sim:
        # (this line runs after the restart when you re-run the cell)

# Try the conda install (idempotent)
try:
    import habitat_sim
    print('habitat_sim ready:', habitat_sim.__version__)
except ImportError:
    print('Running conda install habitat-sim...')
    !conda install -y -c conda-forge -c aihabitat habitat-sim withbullet headless 2>&1 | tail -3
    import habitat_sim
    print('habitat_sim installed:', habitat_sim.__version__)

# Pack ALL Replica scenes (~18 scenes × 50 views ≈ 5 minutes)
!cd /content/trivima && python tools/replica_to_torch.py \
    --replica_root /content/data/Replica \
    --all \
    --views 50 \
    --out_dir /content/data/torch_packed

!ls -lh /content/data/torch_packed/ 2>/dev/null || echo 'no .torch files produced'

## 5. Build the GS-LRM model

In [ ]:
import torch
from trivima.multiview.gs_lrm import GSLRM, build_gaussians, render_gaussians, gs_lrm_loss

device = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

model = GSLRM(img_size=256, decoder_dim=256, depth_min=0.1, depth_max=200.0).to(device)

# Force encoder init via a dummy forward
with torch.no_grad():
    dummy = torch.randn(1, 3, 256, 256, device=device)
    out = model(dummy)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {n_train/1e6:.1f}M / {n_total/1e6:.1f}M total')
print(f'depth shape: {out["depth"].shape}, rgb shape: {out["rgb"].shape}')

## 6. Build the dataset (Phase A: Replica only)

In [ ]:
from torch.utils.data import DataLoader
from trivima.multiview.pointcloud_dataset import MultiDomainDataset, collate_fn

sources = [
    ('/content/data/torch_packed/replica_*.torch', 'indoor'),
    # Add more in Phase B/C/D:
    # ('/content/data/torch_packed/kitti_*.torch', 'outdoor'),
    # ('/content/data/torch_packed/megadepth_*.torch', 'landmark'),
]
dataset = MultiDomainDataset(sources, img_size=256, num_target_views=4)
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    drop_last=True,
    collate_fn=collate_fn,
    pin_memory=True,
    persistent_workers=True,
)
print(f'Batches/epoch: {len(loader)}')

# Sanity check one batch
batch = next(iter(loader))
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: {tuple(v.shape)}  {v.dtype}')
    else:
        print(f'  {k}: {v}')

## 7. Train (auto-resume + Drive save every epoch)

In [ ]:
import time, math, numpy as np
import torch.nn.functional as F

EPOCHS = 200
LR_PEAK = 2e-4
LR_MIN = 1e-6
WARMUP_STEPS = 200
TIME_BUDGET_SEC = 5 * 3600
DEPTH_WEIGHT = 0.1

BEST_PATH   = f'{DRIVE_DIR}/best_gslrm.pt'
RESUME_PATH = f'{DRIVE_DIR}/last_gslrm.pt'

trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR_PEAK, weight_decay=0.01)
best_loss = float('inf')
start_epoch = 0
global_step = 0

if os.path.exists(RESUME_PATH):
    print(f'Resuming from {RESUME_PATH}')
    ck = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(ck['model'], strict=False)
    optimizer.load_state_dict(ck['optimizer'])
    start_epoch = ck['epoch'] + 1
    global_step = ck.get('global_step', 0)
    best_loss = ck.get('best_loss', float('inf'))
    print(f'Resumed epoch={start_epoch} step={global_step} best={best_loss:.4f}')
else:
    print('Fresh run')

total_steps_estimate = EPOCHS * len(loader)
def lr_at(step):
    if step < WARMUP_STEPS:
        return LR_PEAK * (step + 1) / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, total_steps_estimate - WARMUP_STEPS)
    p = min(1.0, p)
    return LR_MIN + 0.5 * (LR_PEAK - LR_MIN) * (1 + math.cos(math.pi * p))

def save(path, epoch, step, loss):
    torch.save({
        'model': {k: v for k, v in model.state_dict().items() if 'encoder' not in k},  # skip frozen DINO
        'optimizer': optimizer.state_dict(),
        'epoch': epoch, 'global_step': step, 'best_loss': loss,
    }, path)

t_start = time.time()
stop = False
for epoch in range(start_epoch, EPOCHS):
    if stop: break
    losses = []
    t0 = time.time()
    model.train()
    for bi, batch in enumerate(loader):
        if time.time() - t_start > TIME_BUDGET_SEC:
            print(f'\nTime budget reached')
            stop = True
            break
        for pg in optimizer.param_groups:
            pg['lr'] = lr_at(global_step)

        inp_img = batch['input_image'].to(device, non_blocking=True)
        inp_K   = batch['input_intrinsics'].to(device, non_blocking=True)
        inp_c2w = batch['input_cam2world'].to(device, non_blocking=True)
        tgt_img = batch['targets'].to(device, non_blocking=True)
        tgt_d   = batch['target_depths'].to(device, non_blocking=True)
        tgt_m   = batch['target_masks'].to(device, non_blocking=True)
        tgt_K   = batch['target_intrinsics'].to(device, non_blocking=True)
        tgt_c2w = batch['target_cam2world'].to(device, non_blocking=True)

        with torch.autocast('cuda', dtype=torch.bfloat16):
            out = model(inp_img)
        out_fp32 = {k: v.float() for k, v in out.items()}
        gaussians = build_gaussians(out_fp32, inp_K, inp_c2w)

        # Render against each target view
        K = tgt_img.shape[1]
        loss_total = 0.0
        loss_rgb_v = 0.0
        loss_d_v = 0.0
        for k in range(K):
            w2c_k = torch.linalg.inv(tgt_c2w[:, k])  # (B, 4, 4)
            rgb_pred, depth_pred = render_gaussians(
                gaussians, w2c_k, tgt_K[:, k], width=256, height=256,
            )
            gt_rgb = tgt_img[:, k].permute(0, 2, 3, 1)  # (B, H, W, 3)
            losses_dict = gs_lrm_loss(
                rgb_pred, depth_pred,
                gt_rgb, tgt_d[:, k], tgt_m[:, k],
                depth_weight=DEPTH_WEIGHT,
            )
            loss_total = loss_total + losses_dict['total']
            loss_rgb_v = loss_rgb_v + losses_dict['rgb'].item()
            loss_d_v = loss_d_v + losses_dict['depth'].item()
        loss_total = loss_total / K

        optimizer.zero_grad(set_to_none=True)
        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        losses.append(loss_total.item())
        global_step += 1
        if (bi+1) % 10 == 0:
            print(f'  [{epoch+1}] {bi+1}/{len(loader)} loss={np.mean(losses[-10:]):.4f} '
                  f'rgb={loss_rgb_v/K:.4f} d={loss_d_v/K:.4f} lr={optimizer.param_groups[0]["lr"]:.2e}')

    if not losses: break
    avg = float(np.mean(losses))
    dt = time.time() - t0
    elapsed = (time.time() - t_start) / 3600
    print(f'Epoch {epoch+1}: loss={avg:.4f} ({dt:.0f}s) total={elapsed:.2f}h')

    save(RESUME_PATH, epoch, global_step, best_loss)
    if avg < best_loss:
        best_loss = avg
        save(BEST_PATH, epoch, global_step, best_loss)
        print(f'  -> new best: {best_loss:.4f}')

print(f'\nDone. Best: {best_loss:.4f}, total: {(time.time()-t_start)/3600:.2f}h')

## 8. Visualize: input → predicted Gaussians → 8 orbit renders

In [ ]:
import matplotlib.pyplot as plt

ck = torch.load(BEST_PATH, map_location=device)
model.load_state_dict(ck['model'], strict=False)
model.eval()

batch = next(iter(loader))
inp_img = batch['input_image'][:1].to(device)
inp_K   = batch['input_intrinsics'][:1].to(device)
inp_c2w = batch['input_cam2world'][:1].to(device)

with torch.no_grad():
    out = model(inp_img)
    gaussians = build_gaussians({k: v.float() for k, v in out.items()}, inp_K, inp_c2w)

    # Render an orbit around the input camera position
    cam_center = inp_c2w[0, :3, 3].cpu().numpy()
    radius = float(batch['scene_scale'][0]) * 0.3
    views = []
    for i in range(8):
        theta = 2 * np.pi * i / 8
        # Build a c2w that orbits cam_center
        c2w = inp_c2w[0].clone()
        c2w[0, 3] += radius * np.sin(theta)
        c2w[2, 3] += radius * (np.cos(theta) - 1)
        w2c = torch.linalg.inv(c2w).unsqueeze(0)
        rgb, _ = render_gaussians(gaussians, w2c, inp_K, 256, 256)
        views.append(rgb[0].cpu().clamp(0, 1).numpy())

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes[0, 0].imshow(inp_img[0].cpu().permute(1, 2, 0).numpy()); axes[0, 0].set_title('INPUT')
for i in range(8):
    r, c = (0, i+1) if i < 4 else (1, i-3)
    axes[r, c].imshow(views[i])
    axes[r, c].set_title(f'orbit {i+1}')
axes[1, 0].axis('off')
for ax in axes.flat: ax.axis('off')
plt.suptitle('GS-LRM: 1 photo → 3D Gaussians → 8 orbit views')
plt.tight_layout()
plt.savefig('/content/gs_lrm_orbit.png', dpi=150)
plt.show()

## 9. PLY export (open in MeshLab to verify geometry)

In [ ]:
from trivima.gaussian.export import export_gaussians_ply

# Convert (B, N, 3) → (N, 3) for the export function
gs_export = {
    'positions': gaussians['positions'][0].cpu(),
    'colors':    gaussians['colors'][0].cpu(),
    'opacities': gaussians['opacities'][0].cpu(),
    'scales':    gaussians['scales'][0].cpu(),
    'rotations': gaussians['rotations'][0].cpu(),
}
export_gaussians_ply(gs_export, '/content/gs_lrm_pred.ply')
!ls -lh /content/gs_lrm_pred.ply

## 10. (Phase B) Add KITTI + MegaDepth — re-run cell 6

Once Phase A confirms the indoor pipeline works, uncomment the additional sources in cell 6 and run from cell 6 again. The MultiDomainDataset will pick up the new .torch files automatically.

Packing scripts:
```
!python tools/kitti_to_torch.py --kitti_root /content/data/kitti_raw \
    --drive 2011_09_26/2011_09_26_drive_0001_sync \
    --frames 100 --out /content/data/torch_packed/kitti_drive_0001.torch

!python tools/megadepth_to_torch.py \
    --scene_root /content/data/MegaDepth_v1/0001/dense0 \
    --colmap_dir /content/data/MegaDepth_SfM/0001/sparse/manhattan/0 \
    --max_views 60 --out /content/data/torch_packed/megadepth_0001.torch
```

The training cell will auto-resume from the Phase A checkpoint.